In [1]:
#Import packages
#---------------------------------------
import sys
import os
import glob
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import matplotlib
import kedm
import warnings
import gc
warnings.filterwarnings("ignore", category=RuntimeWarning) 

#Import your modules
#---------------------------------------
import CCM as cfn
import proc_functions as pfn

sys.path.insert(0, os.path.expanduser('~/Documents/admin_tools'))
from admin_tools import admin_functions as adfn

# Define paths
#----------------------------------------------------------------------
code = '~/Documents/empirical_dynamic_modelling'
data = '/snl/scratch25/dburrows/CCM'

%load_ext autoreload
sys.version

Kokkos::OpenMP::initialize WARNING: OMP_PROC_BIND environment variable not set
  In general, for best performance with OpenMP 4.0 or better set OMP_PROC_BIND=spread and OMP_PLACES=threads
  For best performance with OpenMP 3.1 set OMP_PROC_BIND=true
  For unit testing set OMP_PROC_BIND=false



'3.7.4 (default, Aug 13 2019, 20:35:49) \n[GCC 7.3.0]'

# Load in data

In [2]:
#group data
#================
#group by sample
xmap_l = np.asarray([os.path.basename(i) for i in np.sort(glob.glob(data + '/*xmap*'))])

import re
bln = xmap_l[[bool(re.search(r'(BLN|PTZ05)', i)) for i in xmap_l]]
ptz20 = xmap_l[[bool(re.search(r'(PTZ20)', i)) for i in xmap_l]]

#filter out 12 pre
s = pd.Series(ptz20)  # your series
mask = ~(s.str.contains(r'-12', na=False) & s.str.contains(r'pre', na=False))   # case-sensitive
ptz20 = ptz20[mask]

#unique sample list 
unq_l = np.append(np.unique(np.asarray([re.search(r'^(.*?_run-01)', i).group(1) for i in bln])),
          np.unique(np.asarray([''.join(re.search(r'^(.*?_run-01)(.{4})', i).groups()) for i in ptz20])))
unq_l

array(['PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ20_run-0

In [39]:
# mask data
%autoreload
#Compute fraction of cells that converge, and mean CCM in convergent cells
#==========================================================
from tqdm import tqdm

#parameters
n_int = 3 #expected interval files
int_l=[1.2, 1.4, 1.8] #expected intervals - make sure numbers are complete as in string! 
# -0.2 - i.e. allow a single negative value with <20% of the mean positive increase, to be convergent
ratio_val = -0.2 

results = []
# Loop through each unique base name
for curr in tqdm(unq_l, desc='Processing CCM files'):
    curr_l = glob.glob(f'{data}/{curr}*xmap*')
    og, mask, conv_arr = pfn.conv_check(curr_l=curr_l, n_int=n_int, int_l=int_l, ratio_val=ratio_val)
    np.save(f'{data}/{curr}_xmap_mask.npy', mask)

Processing CCM files: 100%|████████████████████████████████████████████████████████████████████████████████████████| 28/28 [06:57<00:00, 14.92s/it]


In [12]:
mask_l

array(['/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ05_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ20_run-01_sz__xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-04_2photon_sess-01-6dpf_PTZ20_run-01_sz__xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-06_2photon_sess-01-6dpf_BLN_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-06_2photon_sess-01-6dpf_PTZ05_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_BLN_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ05_run-01_xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ20_run-01_sz__xmap_mask.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-08_2photon_sess-01-6dpf_PTZ

In [18]:
trace_l[~(pd.Series(trace_l).str.contains('PTZ20') & pd.Series(trace_l).str.contains('1_reg'))]

array(['/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ05_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ20_run-01_sz_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-04_2photon_sess-01-6dpf_PTZ20_run-01_sz_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-06_2photon_sess-01-6dpf_BLN_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-06_2photon_sess-01-6dpf_PTZ05_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_BLN_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ05_run-01_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ20_run-01_sz_regtrace.npy',
       '/snl/scratch25/dburrows/CCM/PTZ-WILDTYPE-08_2photon_sess-01-6dpf_PTZ20_run-01_pr

In [169]:
#load mask, trace & coords
#===============================
trace_l = np.sort(glob.glob(f'{data}/*regtrace*'))
xmap_l = np.sort(glob.glob(f'{data}/*_CCMxmap*'))
mask_l = np.sort(glob.glob(f'{data}/*mask*'))
coord_l = np.sort(glob.glob(f'{data}/*labcoord*'))

#extract all files
def xtr(string):
    out = string.split('_')[:-1]
    name ='_'.join(out)
    xmap = np.load(f'{name}_CCMxmap.npy')[1:,1:]
    mask = np.load(f'{name}_mask.npy')[1:,1:]
    trace = np.load(f'{name}_regtrace.npy')

    if 'run-01_sz' in name: coord = np.load(f"{name.replace('sz','')}labcoord.npy")
    elif 'run-01_pre' in name: coord = np.load(f"{name.replace('pre','')}labcoord.npy")
    else: coord = np.load(f'{name}_labcoord.npy')
    trace = trace[coord[:,4] != 'nan']
    coord = coord[coord[:,4] != 'nan']

    assert trace.shape[0] == coord.shape[0] == xmap.shape[0] == mask.shape[0], 'shape mismatch'
    dic = {'trace':trace, 'coord':coord, 'mask':mask, 'xmap':xmap}
    return(dic)

In [173]:
dic = xtr(mask_l[0])

In [ ]:
#figure out directionality

# Coarse network analysis

In [ ]:
# come up with plan -> what is of interest -> what is useful, forget zebrafish

# Non-linearity & dimensionality